In [3]:
import pandas as pd
from datetime import date

In [4]:
DATA_DIR = "../data/"
TRIPS_PATH = DATA_DIR + "trips.txt"
ROUTES_PATH = DATA_DIR + "routes.txt"
STOPTIMES_PATH = DATA_DIR + "stop_times.txt"
CALENDAR_PATH = DATA_DIR + "calendar.txt"
CALENDARDATES_PATH = DATA_DIR + "calendar_dates.txt"
STOPS_PATH = DATA_DIR + "stops.txt"

In [5]:
# only keep routes that have type 1xx (train)
# https://developers.google.com/transit/gtfs/reference/extended-route-types
def get_rail_routes(r):
    return r[(r['route_type'] >= 100) & (r['route_type'] < 200)]

routes = pd.read_csv(ROUTES_PATH)
routes = get_rail_routes(routes)
routes

,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type
0,91-10-A-j26-1,78,S10,NaN,S,109
1,91-10-B-j26-1,11,S10,NaN,S,109
2,91-10-C-j26-1,65,S10,NaN,S,109
3,91-10-D-j26-1,11,10,NaN,EXT,117
6,91-11-A-j26-1,11,S11,NaN,S,109
...,...,...,...,...,...,...
3677,93-81-j26-1,137,81,NaN,CC,116
3678,93-82-j26-1,137,82,NaN,CC,116
3680,93-83-j26-1,136,83,NaN,CC,116
3682,93-87-j26-1,129,87,NaN,CC,116


In [6]:
# only keep trips that use previously filtered routes
# (a trip is one instance of a route)
def filter_trips(t, r):
    return t[t['route_id'].isin(r['route_id'])]

trips = pd.read_csv(TRIPS_PATH)
trips = filter_trips(trips, routes)
trips

,route_id,service_id,trip_id,trip_headsign,trip_short_name,direction_id,block_id,original_trip_id,hints
0,91-10-A-j26-1,TA+lbs00,1.TA.91-10-A-j26-1.1.H,Zürich HB,12896,0,NaN,ch:1:sjyid:100058:12896-002,2 NF VN
1,91-10-A-j26-1,TA+lbs00,10.TA.91-10-A-j26-1.1.H,Zürich HB,12824,0,NaN,ch:1:sjyid:100058:12824-002,2 NF
2,91-10-A-j26-1,TA+ral00,100.TA.91-10-A-j26-1.5.H,Zürich Selnau,12792,0,NaN,ch:1:sjyid:100058:12792-002,2 NF
3,91-10-A-j26-1,TA+ral00,101.TA.91-10-A-j26-1.5.H,Zürich Selnau,12816,0,NaN,ch:1:sjyid:100058:12816-003,2 NF
4,91-10-A-j26-1,TA+tal00,102.TA.91-10-A-j26-1.5.H,Zürich Selnau,12884,0,NaN,ch:1:sjyid:100058:12884-003,2 NF
...,...,...,...,...,...,...,...,...,...
1065612,93-88-j26-1,TA,5.TA.93-88-j26-1.3.H,Rigi Staffelhöhe,1103,0,NaN,ch:1:sjyid:100110:SKI-1103,2 NF VN
1065613,93-88-j26-1,TA+jb000,6.TA.93-88-j26-1.4.H,Rigi Kulm,1297,0,NaN,ch:1:sjyid:100110:SKI-1297,2 NF VN
1065614,93-88-j26-1,TA+zP,7.TA.93-88-j26-1.5.R,Vitznau,1202,1,NaN,ch:1:sjyid:100110:SKI-1202,2 NF VN
1065615,93-88-j26-1,TA+bhd00,8.TA.93-88-j26-1.6.R,Vitznau,1264,1,NaN,ch:1:sjyid:100110:SKI-1264,2 NF VN


In [7]:
def weekday_nb_to_column_name(d):
    return ['monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday'][d.weekday()]

# only keep services that run on the given day
def filter_valid_services(s: pd.DataFrame, d: date, e: pd.DataFrame):
    d = pd.to_datetime(d)
    col = weekday_nb_to_column_name(d)
    s = s[(s['start_date'] <= d) & (s['end_date'] >= d)]
    s = s[(s[col] == 1)]
    e = e[e['date'] == d]
    e = e[e['exception_type'] == 2]['service_id']

    s = s[~s['service_id'].isin(e)]

    return s


def filter_trips_for_date(s, t):
    return t[t['service_id'].isin(s['service_id'])]

exceptions = pd.read_csv(CALENDARDATES_PATH)
exceptions['date'] = pd.to_datetime(exceptions['date'], format="%Y%m%d")

services = pd.read_csv(CALENDAR_PATH)
services['start_date'] = pd.to_datetime(services['start_date'], format='%Y%m%d')
services['end_date'] = pd.to_datetime(services['end_date'], format='%Y%m%d')
services = filter_valid_services(services, date.today(), exceptions)

trips = filter_trips_for_date(services, trips)
trips

,route_id,service_id,trip_id,trip_headsign,trip_short_name,direction_id,block_id,original_trip_id,hints
0,91-10-A-j26-1,TA+lbs00,1.TA.91-10-A-j26-1.1.H,Zürich HB,12896,0,NaN,ch:1:sjyid:100058:12896-002,2 NF VN
1,91-10-A-j26-1,TA+lbs00,10.TA.91-10-A-j26-1.1.H,Zürich HB,12824,0,NaN,ch:1:sjyid:100058:12824-002,2 NF
12,91-10-A-j26-1,TA+lbs00,11.TA.91-10-A-j26-1.1.H,Zürich HB,12842,0,NaN,ch:1:sjyid:100058:12842-001,2 NF
23,91-10-A-j26-1,TA+lbs00,12.TA.91-10-A-j26-1.1.H,Zürich HB,12882,0,NaN,ch:1:sjyid:100058:12882-001,2 NF VN
29,91-10-A-j26-1,TA+lbs00,125.TA.91-10-A-j26-1.7.H,Zürich HB,12764,0,NaN,ch:1:sjyid:100058:12764-001,2 NF VN
...,...,...,...,...,...,...,...,...,...
1065608,93-88-j26-1,TA,11.TA.93-88-j26-1.7.R,Vitznau,1104,1,NaN,ch:1:sjyid:100110:SKI-1104,2 NF VN
1065610,93-88-j26-1,TA+bhd00,3.TA.93-88-j26-1.1.H,Rigi Kaltbad-First,1263,0,NaN,ch:1:sjyid:100110:SKI-1263,2 NF VN
1065612,93-88-j26-1,TA,5.TA.93-88-j26-1.3.H,Rigi Staffelhöhe,1103,0,NaN,ch:1:sjyid:100110:SKI-1103,2 NF VN
1065614,93-88-j26-1,TA+zP,7.TA.93-88-j26-1.5.R,Vitznau,1202,1,NaN,ch:1:sjyid:100110:SKI-1202,2 NF VN


In [11]:
# filter stop times
def filter_stop_times(t, st):
    st = st[st['trip_id'].isin(t['trip_id'])]
    max_seq = st.groupby('trip_id')['stop_sequence'].transform('max')
    st = st[st['stop_sequence'] < max_seq]
    return st

stop_times = pd.read_csv(STOPTIMES_PATH)
stop_times = filter_stop_times(trips, stop_times)
stop_times

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,pickup_type,drop_off_type
0,1.TA.91-10-A-j26-1.1.H,17:37:00,17:37:00,8503054:0:1,1,0,0
1,1.TA.91-10-A-j26-1.1.H,17:38:00,17:38:00,8503053:0:1,2,0,0
2,1.TA.91-10-A-j26-1.1.H,17:39:00,17:39:00,8503052:0:1,3,0,0
3,1.TA.91-10-A-j26-1.1.H,17:42:00,17:42:00,8503051:0:1,4,0,0
4,1.TA.91-10-A-j26-1.1.H,17:45:00,17:45:00,8503090:0:1,5,0,0
...,...,...,...,...,...,...,...
17376144,92.TA.93-64-j26-1.11.R,18:03:00,18:06:00,8507376:0:1,4,0,0
17376845,93.TA.93-64-j26-1.11.R,17:01:00,17:01:00,8507374:0:12,1,0,0
17376846,93.TA.93-64-j26-1.11.R,17:15:00,17:15:00,8507375:0:1,2,0,0
17376847,93.TA.93-64-j26-1.11.R,17:21:00,17:21:00,8507552:0:1,3,0,0


In [12]:
def add_parent_station(st, sm):
    st['parent_stop_id'] = st['stop_id'].map(sm)
    return st

stations = pd.read_csv(STOPS_PATH)
stations_map = stations.set_index('stop_id')['parent_station']
stop_times = add_parent_station(stop_times, stations_map)
stop_times

/var/folders/0l/57kb574j2ln24q33sxlqqv280000gn/T/ipykernel_12340/2096769430.py:5: DtypeWarning: Columns (0: parent_station, 1: platform_code) have mixed types. Specify dtype option on import or set low_memory=False.
  stations = pd.read_csv(STOPS_PATH)


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,pickup_type,drop_off_type,parent_stop_id
0,1.TA.91-10-A-j26-1.1.H,17:37:00,17:37:00,8503054:0:1,1,0,0,Parent8503054
1,1.TA.91-10-A-j26-1.1.H,17:38:00,17:38:00,8503053:0:1,2,0,0,Parent8503053
2,1.TA.91-10-A-j26-1.1.H,17:39:00,17:39:00,8503052:0:1,3,0,0,Parent8503052
3,1.TA.91-10-A-j26-1.1.H,17:42:00,17:42:00,8503051:0:1,4,0,0,Parent8503051
4,1.TA.91-10-A-j26-1.1.H,17:45:00,17:45:00,8503090:0:1,5,0,0,Parent8503090
...,...,...,...,...,...,...,...,...
17376144,92.TA.93-64-j26-1.11.R,18:03:00,18:06:00,8507376:0:1,4,0,0,Parent8507376
17376845,93.TA.93-64-j26-1.11.R,17:01:00,17:01:00,8507374:0:12,1,0,0,Parent8507374
17376846,93.TA.93-64-j26-1.11.R,17:15:00,17:15:00,8507375:0:1,2,0,0,Parent8507375
17376847,93.TA.93-64-j26-1.11.R,17:21:00,17:21:00,8507552:0:1,3,0,0,Parent8507552


In [23]:
def get_stop_times_for_station(st: pd.DataFrame, t: pd.DataFrame, r: pd.DataFrame, parent_sid):
    st = st[st['parent_stop_id'] == parent_sid]
    st['headsign'] = st['trip_id'].map(t.set_index('trip_id')['trip_headsign'])
    st['line'] = st['trip_id'].map(t.set_index('trip_id')['route_id']).map(r.set_index('route_id')['route_short_name'])
    st = st.sort_values(by=['departure_time'])
    st = st.reset_index()
    return st


def print_departure_board(st: pd.DataFrame, start_hour, limit):
    st = st[st['departure_time'] > start_hour]
    limit = min(limit, st.shape[0])
    for i in range(limit):
        h = st.iloc[i]['departure_time'][:-3]
        line = st.iloc[i]['line']
        hs = st.iloc[i]['headsign']
        print(f"{h} {line:5} {hs}")


renens = 'Parent8501118'
dep = get_stop_times_for_station(stop_times, trips, routes, renens)[['departure_time', 'line', 'headsign']]
print_departure_board(dep, '09:00:00', 10)

09:04 IR95  Brig
09:04 R3    Vallorbe
09:06 RE33  Annemasse
09:08 IC1   St. Gallen
09:09 R9    Allaman
09:11 IC51  Basel SBB
09:16 R1    Grandson
09:16 R2    Bex
09:20 IC5   Lausanne
09:21 R8    Lausanne
